In [1]:
import requests
import json
import pandas as pd
import os
import tqdm as tqdm
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote_plus
import rdflib
import os
from rdflib.namespace import RDF, DC, Namespace
import xml.etree.ElementTree as ET
from lxml import etree

In [11]:
def parse_rdf_filess(directory):
    # Define the namespaces
    namespaces = {
        'rdf': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
        'dc': 'http://purl.org/dc/elements/1.1/',
        'dcterms': 'http://purl.org/dc/terms/',
        'edm': 'http://www.europeana.eu/schemas/edm/',
        'skos': 'http://www.w3.org/2004/02/skos/core#',
        'foaf': 'http://xmlns.com/foaf/0.1/',
        'ore': 'http://www.openarchives.org/ore/terms/',
        'dqv': 'http://www.w3.org/ns/dqv#',
        'oa': 'http://www.w3.org/ns/oa#'
    }

    # Define functions to find elements
    def find_single_element_text(tree, xpath_query):
        element = tree.find(xpath_query, namespaces)
        if element is not None:
            if element.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource'):
                return element.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource')
            else:
                return element.text
        return None

    def find_multiple_elements_text(tree, xpath_query):
        elements = tree.findall(xpath_query, namespaces)
        return [element.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource') or element.text 
                for element in elements if element is not None and xpath_query != '//dc:date']
    
    def find_tier_information(tree, namespaces):
        # Initialize variables to store tier information
        content_tier = None
        metadata_tier = None

        # Find all hasBody elements
        has_body_elements = tree.findall('.//oa:hasBody', namespaces)

        for element in has_body_elements:
            resource = element.get('{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource', '')
            
            # Check for content tier
            if 'contentTier' in resource:
                content_tier = resource.split('contentTier')[-1]
            
            # Check for metadata tier
            elif 'metadataTier' in resource:
                metadata_tier = resource.split('metadataTier')[-1]

        return content_tier, metadata_tier

    # Start building the Solr input document
    solr_docs = "<add>"

    # Iterate over each file in the directory
    for filename in os.listdir(directory):
        if filename.endswith('.rdf') or filename.endswith('.xml'):
            file_path = os.path.join(directory, filename)
            
            try:
                # Load the XML file
                tree = etree.parse(file_path)
                print(f'Parsed: {filename}')

                # Extract data
                data = {
                    'ore:proxyIn' : find_single_element_text(tree, '//ore:proxyIn'),
                    'edm:dataProvider': find_single_element_text(tree, '//edm:dataProvider'),
                    'edm:intermediateProvider': find_single_element_text(tree, '//edm:intermediateProvider'),
                    'edm:provider': find_single_element_text(tree, '//ore:Aggregation/edm:provider'),
                    'edm:type': find_single_element_text(tree, '//edm:type'),
                    'edm:currentLocation': find_single_element_text(tree, '//edm:currentLocation'),
                    'dc:contributor': find_multiple_elements_text(tree, '//dc:contributor'),
                    'dc:coverage': find_single_element_text(tree, '//dc:coverage'),
                    'dc:creator': find_single_element_text(tree, '//dc:creator'),
                    'dc:date': find_single_element_text(tree, '//dc:date'),
                    'dc:description': find_single_element_text(tree, '//dc:description'),
                    'dc:format': find_single_element_text(tree, '//dc:format'),
                    'dc:language': find_single_element_text(tree, '//dc:language'),
                    'dc:publisher': find_single_element_text(tree, '//dc:publisher'),
                    'dc:source': find_single_element_text(tree, '//dc:source'),
                    'dc:subject': find_single_element_text(tree, '//dc:subject'),
                    'dc:title': find_single_element_text(tree, '//dc:title'),
                    'dc:type': find_single_element_text(tree, '//dc:type'),
                    'dcterms:alternative': find_single_element_text(tree, '//dcterms:alternative'),
                    'dcterms:created': find_single_element_text(tree, '//dcterms:created'),
                    'dcterms:issued': find_single_element_text(tree, '//dcterms:issued'),
                    'dcterms:medium': find_multiple_elements_text(tree, '//dcterms:medium'),
                    'dcterms:provenance': find_single_element_text(tree, '//dcterms:provenance'),
                    'dcterms:spatial': find_single_element_text(tree, '//dcterms:spatial'),
                    'dcterms:temporal': find_single_element_text(tree, '//dcterms:temporal'),
                    'dcterms:modified': find_single_element_text(tree, '//dcterms:modified'),
                    'skos:prefLabel': find_multiple_elements_text(tree, '//skos:prefLabel'),
                    'skos:altLabel': find_multiple_elements_text(tree, '//skos:altLabel'),
                }
                content_tier, metadata_tier = find_tier_information(tree, namespaces)
                print(f'Content Tier: {content_tier}, Metadata Tier: {metadata_tier}')
                # Build Solr document
                solr_docs += f"""
                <doc>
                    <field name="europeana_id">{data['ore:proxyIn']} </field>
                    <field name="proxy_dc_title">{data['dc:title']} </field>
                    <field name="proxy_dc_creator">{data['dc:creator']} </field>
                    <field name="proxy_dc_date">{data['dc:date']} </field>
                    <field name="edm_dataProvider">{data['edm:dataProvider']} </field>
                    <field name="edm_intermediateProvider">{data['edm:intermediateProvider']} </field>
                    <field name="edm_provider">{data['edm:provider']} </field>
                    <field name="proxy_dc_contributor">{data['dc:contributor']} </field>
                    <field name="dcterms_created">{data['dcterms:created']} </field>
                    <field name="edm_type">{data['edm:type']} </field>
                    <field name="skos_prefLabel">{data['skos:prefLabel']} </field>
                    <field name="skos_altLabel">{data['skos:altLabel']} </field>
                    <field name="dc_description">{data['dc:description']} </field>
                    <field name="dc_format">{data['dc:format']} </field>
                    <field name="dc_language">{data['dc:language']} </field>
                    <field name="dc_publisher">{data['dc:publisher']} </field>
                    <field name="dc_source">{data['dc:source']} </field>
                    <field name="dc_subject">{data['dc:subject']} </field>
                    <field name="dc_type">{data['dc:type']} </field>
                    <field name="dcterms_alternative">{data['dcterms:alternative']} </field>
                    <field name="dcterms_issued">{data['dcterms:issued']} </field>
                    <field name="dcterms_medium">{data['dcterms:medium']} </field>
                    <field name="dcterms_provenance">{data['dcterms:provenance']} </field>
                    <field name="dcterms_spatial">{data['dcterms:spatial']} </field>
                    <field name="dcterms_temporal">{data['dcterms:temporal']} </field>
                    <field name="dc_coverage">{data['dc:coverage']} </field>
                    <field name="edm_currentLocation">{data['edm:currentLocation']} </field>
                    <field name="dcterms_modified">{data['dcterms:modified']} </field>
                    <field name="content_tier">{content_tier} </field>
                    <field name="metadata_tier">{metadata_tier} </field>
                </doc>
                """

            except Exception as e:
                print(f"Failed to parse {filename}: {e}")
                continue

    # Close the Solr document
    solr_docs += "</add>"
    
    # Return the compiled Solr XML document
    return solr_docs

In [13]:
# Usage
directory = '/Users/suhaibbasir/Documents/CS/MSc/Thesis/Thesis/EDP/2021672'  # Change this to your directory containing RDF/XML files
solr_xml_data = parse_rdf_filess(directory)
print(solr_xml_data)

Parsed: resource_document_mauritshuis_397.xml
Content Tier: 1, Metadata Tier: A
Parsed: resource_document_mauritshuis_340.xml
Content Tier: 4, Metadata Tier: A
Parsed: resource_document_mauritshuis_426.xml
Content Tier: 4, Metadata Tier: A
Parsed: resource_document_mauritshuis_432.xml
Content Tier: 4, Metadata Tier: A
Parsed: resource_document_mauritshuis_354.xml
Content Tier: 4, Metadata Tier: A
Parsed: resource_document_mauritshuis_181.xml
Content Tier: 4, Metadata Tier: A
Parsed: resource_document_mauritshuis_195.xml
Content Tier: 4, Metadata Tier: A
Parsed: resource_document_mauritshuis_1159.xml
Content Tier: 4, Metadata Tier: A
Parsed: resource_document_mauritshuis_803.xml
Content Tier: 1, Metadata Tier: A
Parsed: resource_document_mauritshuis_1171.xml
Content Tier: 4, Metadata Tier: A
Parsed: resource_document_mauritshuis_9.xml
Content Tier: 4, Metadata Tier: A
Parsed: resource_document_mauritshuis_817.xml
Content Tier: 4, Metadata Tier: A
Parsed: resource_document_mauritshuis_11

In [14]:
def write_solr_xml(data, output_file):
    # Write the data to an XML file
    with open(output_file, 'w') as file:
        file.write(data)
    print(f"Data written to {output_file}")

def remove_duplicates(xml_data):
    # Parse the XML data
    root = ET.fromstring(xml_data)

    # Initialize a set to track unique europeana_id
    seen_ids = set()
    unique_docs = []

    # Iterate over each document and filter out duplicates
    for doc in root.findall(".//doc"):
        europeana_id = doc.find(".//field[@name='europeana_id']").text
        if europeana_id not in seen_ids:
            seen_ids.add(europeana_id)
            unique_docs.append(doc)

    # Build a new XML tree with unique documents
    new_root = ET.Element("add")
    for doc in unique_docs:
        new_root.append(doc)

    # Convert the tree back to a string
    new_xml_data = ET.tostring(new_root, encoding='unicode')
    return new_xml_data

In [15]:
solr_xml_data = remove_duplicates(solr_xml_data)
output_file = '/Users/suhaibbasir/Documents/CS/MSc/Thesis/Thesis/EDP/output_solr_test.xml'
write_solr_xml(solr_xml_data, output_file)

Data written to /Users/suhaibbasir/Documents/CS/MSc/Thesis/Thesis/EDP/output_solr_test.xml
